# Experiment 3: Explainable AI (XAI)

Applies SHAP and LIME on the best model from Exp 1 + 2.

Outputs:
- SHAP global feature importance (bar chart)
- SHAP beeswarm plot
- LIME per-sample explanation
- (Optional) Attention weight heatmap if Transformer was trained

All plots saved as PNG to `exp3_results/`.

In [ ]:
# ============================================================
# CONFIG
# ============================================================
TARGET_DURATION   = 6      # match the best duration from Exp 1
BEST_CLASSIFIER   = 'RF'   # options: 'MLP', 'SVM', 'RF', 'XGB'
                            # LSTM/Transformer use attention viz separately

# Number of samples for SHAP/LIME (fewer = faster)
SHAP_BACKGROUND_SAMPLES = 100   # background samples for SHAP
LIME_SAMPLE_IDX         = 0     # which test sample to explain with LIME
LIME_NUM_FEATURES       = 20    # top N features to show in LIME

# Explain attention for Transformer too?
RUN_ATTENTION_VIZ = False   # set True if Transformer was trained in Exp 2

print('Config loaded.')

In [ ]:
# Install if needed
import subprocess, sys
for pkg in ['shap', 'lime']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])

import os, json, torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving plots
import matplotlib.pyplot as plt
import shap
from lime import lime_tabular

CWD        = os.getcwd()
EXP2_DIR   = os.path.join(CWD, 'exp2_results', f'{TARGET_DURATION}s')
EXP1_DIR   = os.path.join(CWD, 'exp1_results', f'{TARGET_DURATION}s')
OUTPUT_DIR = os.path.join(CWD, 'exp3_results', f'{TARGET_DURATION}s_{BEST_CLASSIFIER}')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

## Step 1: Load Features and Best Model

In [ ]:
# Load pooled features from Exp 1
tr = torch.load(os.path.join(EXP1_DIR, 'pooled_train.pt'))
vl = torch.load(os.path.join(EXP1_DIR, 'pooled_val.pt'))
te = torch.load(os.path.join(EXP1_DIR, 'pooled_test.pt'))

train_X, train_y = tr['features'], tr['labels']
val_X,   val_y   = vl['features'], vl['labels']
test_X,  test_y  = te['features'], te['labels']

# Feature names: wav2vec embedding dimensions
feature_names = [f'feat_{i}' for i in range(train_X.shape[1])]

print(f'train: {train_X.shape}, val: {val_X.shape}, test: {test_X.shape}')
print(f'Feature dims: {train_X.shape[1]}')

In [ ]:
import joblib

if BEST_CLASSIFIER == 'RF':
    model = joblib.load(os.path.join(EXP2_DIR, 'rf_model.pkl'))
    print('Loaded Random Forest model.')
    model_type = 'tree'

elif BEST_CLASSIFIER == 'XGB':
    import xgboost as xgb
    model = xgb.XGBClassifier()
    model.load_model(os.path.join(EXP2_DIR, 'xgb_model.json'))
    print('Loaded XGBoost model.')
    model_type = 'tree'

elif BEST_CLASSIFIER == 'SVM':
    model  = joblib.load(os.path.join(EXP2_DIR, 'svm_model.pkl'))
    scaler = joblib.load(os.path.join(EXP2_DIR, 'scaler.pkl'))
    train_X = scaler.transform(train_X)
    val_X   = scaler.transform(val_X)
    test_X  = scaler.transform(test_X)
    print('Loaded SVM + scaler.')
    model_type = 'kernel'

elif BEST_CLASSIFIER == 'MLP':
    class ResearchMLP(nn.Module):
        def __init__(self, input_size, hidden_sizes, output_size, dropout):
            super().__init__()
            layers = []
            prev = input_size
            for i, h in enumerate(hidden_sizes):
                layers += [nn.Linear(prev, h), nn.ReLU()]
                if i < len(hidden_sizes) - 1:
                    layers.append(nn.Dropout(dropout))
                prev = h
            layers.append(nn.Linear(prev, output_size))
            self.network = nn.Sequential(*layers)
        def forward(self, x): return self.network(x)

    model = ResearchMLP(768, [384, 192, 96], 1, 0.3)
    model.load_state_dict(torch.load(os.path.join(EXP2_DIR, 'mlp_best.pth')))
    model.eval()
    print('Loaded MLP model.')
    model_type = 'mlp'

## Step 2: SHAP Analysis

SHAP explains which of the 768 wav2vec embedding dimensions push the model toward REAL or FAKE.

In [ ]:
# Use a small background set (representative subset of training data)
np.random.seed(42)
bg_idx = np.random.choice(len(train_X), size=min(SHAP_BACKGROUND_SAMPLES, len(train_X)), replace=False)
X_background = train_X[bg_idx]
X_explain    = test_X  # explain test set

print(f'Background: {X_background.shape}')
print(f'Explain:    {X_explain.shape}')

In [ ]:
print(f'Creating SHAP explainer for {BEST_CLASSIFIER}...')

if model_type == 'tree':
    # TreeExplainer: fast and exact for RF/XGB
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_explain)
    # RF returns list [class_0_shap, class_1_shap]; take class 1 (fake)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    print(f'SHAP values shape: {np.array(shap_values).shape}')

elif model_type == 'kernel':
    # KernelExplainer: model-agnostic, slower
    def svm_predict_proba(X):
        return model.predict_proba(X)[:, 1]
    explainer   = shap.KernelExplainer(svm_predict_proba, X_background)
    shap_values = explainer.shap_values(X_explain[:50], nsamples=100)  # limit for speed
    print(f'SHAP values shape: {np.array(shap_values).shape}')

elif model_type == 'mlp':
    # DeepExplainer for PyTorch
    bg_tensor  = torch.FloatTensor(X_background)
    ex_tensor  = torch.FloatTensor(X_explain)
    explainer  = shap.DeepExplainer(model, bg_tensor)
    shap_values = explainer.shap_values(ex_tensor)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.array(shap_values)
    print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# --- Plot 1: Global feature importance (top 20 dimensions) ---
mean_abs_shap = np.abs(shap_values).mean(axis=0)  # [768]
top20_idx  = np.argsort(mean_abs_shap)[::-1][:20]
top20_vals = mean_abs_shap[top20_idx]
top20_names = [feature_names[i] for i in top20_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(20), top20_vals[::-1])
ax.set_yticks(range(20))
ax.set_yticklabels(top20_names[::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title(f'Top 20 Most Important wav2vec Dimensions ({BEST_CLASSIFIER})')
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'shap_top20_importance.png')
plt.savefig(path, dpi=150)
plt.close()
print(f'Saved: {path}')

In [ ]:
# --- Plot 2: SHAP beeswarm (summary plot) ---
# Shows distribution of SHAP values across all test samples for top features
shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=np.zeros(len(shap_values)),
    data=X_explain if model_type != 'mlp' else X_explain,
    feature_names=feature_names
)

plt.figure()
shap.plots.beeswarm(shap_explanation, max_display=20, show=False)
plt.title(f'SHAP Beeswarm: {BEST_CLASSIFIER}')
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'shap_beeswarm.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {path}')

In [ ]:
# --- Plot 3: SHAP waterfall for a single sample ---
# Waterfall shows how each feature pushes one prediction from base to final
single_shap = shap.Explanation(
    values=shap_values[LIME_SAMPLE_IDX],
    base_values=float(np.mean(shap_values)),
    data=X_explain[LIME_SAMPLE_IDX],
    feature_names=feature_names
)

true_label = 'FAKE' if int(test_y[LIME_SAMPLE_IDX]) == 1 else 'REAL'

plt.figure()
shap.plots.waterfall(single_shap, max_display=15, show=False)
plt.title(f'SHAP Waterfall: sample {LIME_SAMPLE_IDX} (true label: {true_label})')
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, f'shap_waterfall_sample{LIME_SAMPLE_IDX}.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {path}')

## Step 3: LIME Analysis

LIME explains individual predictions by fitting a local linear model around each sample.

In [ ]:
print('Creating LIME explainer...')

# Define predict function that returns [prob_real, prob_fake]
if model_type in ['tree', 'kernel']:
    def predict_fn(X):
        proba = model.predict_proba(X)
        return proba  # [N, 2]

elif model_type == 'mlp':
    def predict_fn(X):
        with torch.no_grad():
            logits = model(torch.FloatTensor(X)).squeeze(1)
            prob_fake = torch.sigmoid(logits).numpy()
            prob_real = 1 - prob_fake
        return np.stack([prob_real, prob_fake], axis=1)

lime_explainer = lime_tabular.LimeTabularExplainer(
    train_X,
    feature_names=feature_names,
    class_names=['real', 'fake'],
    mode='classification',
    random_state=42
)
print('LIME explainer ready.')

In [ ]:
# Explain a single test sample
sample      = test_X[LIME_SAMPLE_IDX]
true_label  = 'FAKE' if int(test_y[LIME_SAMPLE_IDX]) == 1 else 'REAL'

print(f'Explaining sample {LIME_SAMPLE_IDX} (true: {true_label})...')

explanation = lime_explainer.explain_instance(
    sample,
    predict_fn,
    num_features=LIME_NUM_FEATURES,
    num_samples=1000
)

# Save LIME plot
fig = explanation.as_pyplot_figure(label=1)  # label=1 -> explains "fake" class
plt.title(f'LIME Explanation: sample {LIME_SAMPLE_IDX} (true: {true_label})')
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, f'lime_sample{LIME_SAMPLE_IDX}.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {path}')

# Print top features
print('\nTop features (feature, weight):')
for feat, weight in explanation.as_list(label=1)[:10]:
    direction = '-> FAKE' if weight > 0 else '-> REAL'
    print(f'  {feat:40s}  {weight:+.4f}  {direction}')

In [ ]:
# Explain multiple samples and aggregate
NUM_LIME_SAMPLES = 10  # how many test samples to explain
print(f'Explaining {NUM_LIME_SAMPLES} test samples for aggregate analysis...')

all_lime_weights = {fn: [] for fn in feature_names}

for idx in range(min(NUM_LIME_SAMPLES, len(test_X))):
    exp = lime_explainer.explain_instance(
        test_X[idx], predict_fn,
        num_features=LIME_NUM_FEATURES, num_samples=500
    )
    for feat, weight in exp.as_list(label=1):
        # feat is a string like 'feat_123 > 0.5', extract feature name
        feat_name = feat.split(' ')[0]
        if feat_name in all_lime_weights:
            all_lime_weights[feat_name].append(abs(weight))
    print(f'  Sample {idx} done.')

# Average absolute LIME weight per feature
lime_importance = {k: np.mean(v) for k, v in all_lime_weights.items() if len(v) > 0}
top_lime = sorted(lime_importance.items(), key=lambda x: x[1], reverse=True)[:20]

# Plot aggregate LIME importance
names  = [x[0] for x in top_lime[::-1]]
values = [x[1] for x in top_lime[::-1]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(names)), values)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Mean |LIME weight|')
ax.set_title(f'Top 20 Features by LIME Importance ({BEST_CLASSIFIER})')
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'lime_aggregate_importance.png')
plt.savefig(path, dpi=150)
plt.close()
print(f'Saved: {path}')

## Step 4: (Optional) Transformer Attention Visualization

If Transformer was trained in Exp 2, visualize attention weights over time frames.
Shows which parts of the audio the model focuses on.

In [ ]:
if RUN_ATTENTION_VIZ:
    print('Loading Transformer model for attention visualization...')

    class TransformerClassifierViz(nn.Module):
        """Same as Exp 2 but stores attention weights during forward pass."""
        def __init__(self, d_model=768, nhead=8, num_layers=2, dropout=0.1):
            super().__init__()
            self.attention_weights = []  # store here during forward
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead,
                dim_feedforward=1024, dropout=dropout,
                batch_first=True
            )
            self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
            self.classifier  = nn.Linear(d_model, 1)

        def forward(self, x):
            out = self.transformer(x)
            out = out.mean(dim=1)
            return self.classifier(out)

    attn_model = TransformerClassifierViz()
    attn_model.load_state_dict(torch.load(os.path.join(EXP2_DIR, 'attn_best.pth')))
    attn_model.eval()
    print('Transformer loaded.')

    # Register hooks to capture attention weights
    attn_maps = []
    def hook_fn(module, input, output):
        attn_maps.append(output)

    hooks = []
    for layer in attn_model.transformer.layers:
        hooks.append(layer.self_attn.register_forward_hook(hook_fn))

    # Load LSTM-level features (frame-level)
    lstm_cache_test = os.path.join(EXP2_DIR, 'lstm_features_test.pt')
    if os.path.exists(lstm_cache_test):
        lstm_test = torch.load(lstm_cache_test)
        X_frame = lstm_test['features']
        y_frame = lstm_test['labels']

        def pad_frames(tensor_list, T_max=None):
            seqs = []
            for t in tensor_list:
                if isinstance(t, list): t = t[0]
                if t.ndim == 3: t = t.squeeze(0)
                seqs.append(t)
            if T_max is None: T_max = max(s.shape[0] for s in seqs)
            padded = []
            for s in seqs:
                T_cur = s.shape[0]
                if T_cur >= T_max: padded.append(s[:T_max])
                else:
                    pad = torch.zeros(T_max - T_cur, s.shape[1])
                    padded.append(torch.cat([s, pad], dim=0))
            return torch.stack(padded), T_max

        X_frame_t, T_MAX = pad_frames(X_frame)

        # Get attention for 1 sample
        sample_input = X_frame_t[LIME_SAMPLE_IDX].unsqueeze(0)  # [1, T, 768]
        attn_maps.clear()
        with torch.no_grad():
            _ = attn_model(sample_input)

        # Remove hooks
        for h in hooks: h.remove()

        # Plot attention weights from each layer
        for layer_idx, attn_out in enumerate(attn_maps):
            if isinstance(attn_out, tuple):
                attn_w = attn_out[1]  # (output, attn_weights)
            else:
                continue
            if attn_w is None:
                print(f'  Layer {layer_idx}: attention weights not returned. Set need_weights=True.')
                continue
            # attn_w shape: [1, T, T]
            attn_w = attn_w.squeeze(0).numpy()  # [T, T]
            # Average over query dimension -> [T] -- attention each frame receives
            attn_received = attn_w.mean(axis=0)

            fig, ax = plt.subplots(figsize=(14, 2))
            ax.plot(attn_received)
            ax.set_xlabel('Time frame')
            ax.set_ylabel('Avg attention weight')
            true_label = 'FAKE' if int(y_frame[LIME_SAMPLE_IDX]) == 1 else 'REAL'
            ax.set_title(f'Attention Layer {layer_idx+1} | sample {LIME_SAMPLE_IDX} ({true_label})')
            plt.tight_layout()
            path = os.path.join(OUTPUT_DIR, f'attn_layer{layer_idx+1}_sample{LIME_SAMPLE_IDX}.png')
            plt.savefig(path, dpi=150)
            plt.close()
            print(f'Saved: {path}')
    else:
        print('LSTM/frame-level features not found. Skipping attention viz.')
        print('Run Exp 2 with RUN_LSTM=True first.')
else:
    print('Attention visualization skipped (RUN_ATTENTION_VIZ=False).')

## Step 5: Save XAI Summary

In [ ]:
# Save SHAP importance to JSON for reporting
top50_shap = [
    {'feature': feature_names[i], 'mean_abs_shap': float(mean_abs_shap[i])}
    for i in np.argsort(mean_abs_shap)[::-1][:50]
]

xai_summary = {
    'classifier'          : BEST_CLASSIFIER,
    'target_duration'     : TARGET_DURATION,
    'feature_dims'        : int(train_X.shape[1]),
    'shap_top50_features' : top50_shap,
    'lime_top_features'   : [
        {'feature': name, 'mean_abs_lime_weight': float(val)}
        for name, val in top_lime
    ]
}

summary_path = os.path.join(OUTPUT_DIR, 'xai_summary.json')
with open(summary_path, 'w') as f:
    json.dump(xai_summary, f, indent=4)

print(f'XAI summary saved to: {summary_path}')
print()
print('=== ALL OUTPUTS ===')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:<45} {size_kb:.1f} KB')